# Datamine PLTLAY Process Example

This notebook demonstrates how to configure and run the **`pltlay`** process wrapper in `dmstudio`.

## Process Description

## PLTLAY

Interactive graphics plot editing and layout process.

There are a number of line, box, circle and text generation sub-commands available in **PLTLAY** in addition to icon storage and retrieval of selected plotted entities.

Additional data overlay plotting functions are available in the **PLTLAY** process. However, in addition to the normal plot file fields the plot file must also contain the implicit fields **XCENTRE, YCENTRE, ZCENTRE, SAZI** and **SDIP**. The default values of these fields, together with the default values of **XMIN, XMAX, YMIN** and **YMAX** fields completely define the position, orientation and size of the viewing plane that will be used in the **PLTLAY** process.

### Input Files:

* **proto** (Plot Prototype):
  Plot prototype.
  Required=No

* **icon** (Plot):
  Icon file. An icon is a small number of plot file records that describe some feature
  that is commonly required on mine plans, e.g. mine shafts. This input/output icon file
  may contain a number of user-defined icons. In addition to the normal plot file fields,
  the icon file will contain the explicit fields **IVALUE, ITEXT, IXSIZE** and **IYSIZE**.
  Required=No

### Output Files:

* **plot** (Plot):
  Output plot file. This file will contain all of the plot data that has been generated
  during the current operation of **PLTLAY**.
  Required=Yes

### Fields:

### Parameters:

* **asize**:
  Type of A size paper, for initial plot size if no prototype file supplied. (0)
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **xscale**:
  Initial X plot scale factor if no prototype file supplied. For example, enter 1000 for
  a scale of 1:1000. Note that user data units of metres are assumed; if metres are not
  the unit, then the scale must be multiplied by factor f, where f=no. of metres in 1 user
  data unit [e.g. 0.3048 for feet].
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **yscale**:
  Initial Y plot scale factor if no prototype file supplied.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **unit**:
  This parameter indicates the type of data that will be brought into the process. The
  default is metric (0) and a unit value of 1 indicates user units of imperial feet.
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **append**:
  If an input plot prototype file has been supplied, any plot records in this file may
  automatically copied to the final output plot file by setting this parameter to 1.(1)
  Range=0,1
  Values=0,1
  Default=1
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('pltlay')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.initialize_sandbox(notebook_folder, files_to_copy=files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute pltlay
print("Running pltlay...")
dm_cmd.pltlay(
    plot_o='t_pltlay_out',  # required
    # proto_i='t_mod1',  # optional
    # icon_i='optional',  # optional
    # asize_p=0,  # optional
    # xscale_p='optional',  # optional
    # yscale_p='optional',  # optional
    # unit_p=0,  # optional
    # append_p=1,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("pltlay execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_pltlay_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")